In [1]:
import pandas as pd
import numpy as np
import stanza
from sentence_transformers import SentenceTransformer

c:\Users\marek\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Inicializácia Stanza
stanza.download('sk')
nlp = stanza.Pipeline(lang='sk', processors='tokenize')

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

2025-06-03 12:58:16 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-03 12:58:16 INFO: Downloading default packages for language: sk (Slovak) ...
2025-06-03 12:58:16 INFO: File exists: C:\Users\marek\stanza_resources\sk\default.zip
2025-06-03 12:58:18 INFO: Finished downloading models and saved to C:\Users\marek\stanza_resources
2025-06-03 12:58:18 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-06-03 12:58:18 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-03 12:58:18 WARNING: Language sk package default expects mwt, which has been added
2025-06-03 12:58:18 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-06-03 12:58:18 INFO: Using device: cpu
2025-06-03 12:58:18 INFO: 

In [3]:
df = pd.read_csv("data/contract_criteria_clean_split.csv")

In [ ]:
# Inicializácia Stanza
stanza.download('sk')
nlp = stanza.Pipeline(lang='sk', processors='tokenize')

# Načítanie modelu a dát
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Výber textových kritérií
all_criteria = df["criterion"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings = model.encode(all_criteria, convert_to_tensor=False)

# Uloženie
np.save("embeddings/criteria_embeddings_split.npy", all_embeddings)
df.to_csv("data/criteria_data.csv", index=False)

2025-06-03 12:58:44 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-03 12:58:44 INFO: Downloading default packages for language: sk (Slovak) ...
2025-06-03 12:58:45 INFO: File exists: C:\Users\marek\stanza_resources\sk\default.zip
2025-06-03 12:58:46 INFO: Finished downloading models and saved to C:\Users\marek\stanza_resources
2025-06-03 12:58:46 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-06-03 12:58:46 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-03 12:58:46 WARNING: Language sk package default expects mwt, which has been added
2025-06-03 12:58:46 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-06-03 12:58:46 INFO: Using device: cpu
2025-06-03 12:58:46 INFO: 

In [ ]:
# Výber textových popisov
all_descriptions = df["description"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings_d = model.encode(all_descriptions, convert_to_tensor=True)

# Uloženie
np.save("embeddings/descriptions_embeddings_split.npy", all_embeddings_d)
df.to_csv("data/descriptions_data.csv", index=False)

In [6]:
import pandas as pd
from transformers import pipeline

# Načítanie dát
df = pd.read_csv("data/contract_criteria_clean_split.csv")

# Pipeline na generovanie textu
generator = pipeline("text2text-generation", model="google/flan-t5-base", max_length=20)

# Pomocná funkcia na vytvorenie kategórie
def create_category(criterion, description):
    prompt = (
        f"Zadaj stručnú všeobecnú kategóriu pre nasledujúce kritérium verejného obstarávania:\n"
        f"Kritérium: {criterion}\n"
        f"Popis: {description}\n"
        f"Kategória:"
    )
    response = generator(prompt, do_sample=False)[0]["generated_text"]
    return response.strip().replace("Kategória:", "").strip().capitalize()

# Vygeneruj kategórie
df["kategoria"] = df.apply(lambda row: create_category(row["criterion"], row["description"]), axis=1)

# Ulož výstup
df.to_csv("data/contract_criteria_with_categories.csv", index=False)
print("✅ Kategórie boli vygenerované a uložené.")

c:\Users\marek\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\marek\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marek\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Deve

✅ Kategórie boli vygenerované a uložené.


In [7]:
# Zoskup a spočítaj kategórie
category_summary = df["kategoria"].value_counts()
print("📚 Vygenerované kategórie:")
print(category_summary)


📚 Vygenerované kategórie:
kategoria
                                 1163
Zdaj strun ve kategoriu           197
Ekn ve vechny kate                184
Zdaj strun ve kategór             170
Ftir ftir ftir ftir ftir ftir     102
                                 ... 
Kod ve ve ve                        1
Kárn kategoriu pedstav              1
Kodn kategorie pedstav              1
Ekn vem kategorie v                 1
Iv.                                 1
Name: count, Length: 379, dtype: int64
